In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Paper V2 — gaussian_shading

独立baseline全量正式任务，使用该方法自己的新校准阈值。

总实验标识：`paper-main-reconstruction-v2`。本notebook固定单个worker，直接运行formal；无任务选择器。代码固定为已完成主方法真实预检的main版本，旧v1结果保留。

In [ ]:
import json, os, pathlib, subprocess, sys
from importlib.metadata import version, PackageNotFoundError

REPO='https://github.com/RICHAAARC/CEG-WM.git'
EXPECTED_EXACT='d9ef4adb4f8424f1102f577bb76347fbba112bd9'
EXPERIMENT_ID='paper-main-reconstruction-v2'
WORKER='gaussian_shading'
checkout=pathlib.Path('/content/cegwm-paper-v2-gaussian_shading')
runtime_root=pathlib.Path('/content/cegwm-paper-v2-runtime')
drive_parent=pathlib.Path('/content/drive/MyDrive/CEG-WM')
drive_root=drive_parent/EXPERIMENT_ID
if not checkout.exists(): subprocess.run(['git','clone','--branch','main','--single-branch',REPO,str(checkout)],check=True)
subprocess.run(['git','-C',str(checkout),'fetch','origin','main'],check=True)
subprocess.run(['git','-C',str(checkout),'checkout','--detach',EXPECTED_EXACT],check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers<0.40', 'transformers', 'accelerate', 'lpips', 'torchmetrics'], check=True)
environment={}
for package in ('torch','diffusers','transformers','accelerate'):
    try: environment[package]=version(package)
    except PackageNotFoundError: environment[package]=None
print({'experiment':EXPERIMENT_ID,'worker':WORKER,'code':EXPECTED_EXACT,'versions':environment})
child_env=dict(os.environ)
child_env['PYTHONPATH']=str(checkout/'src')+os.pathsep+str(checkout)
from google.colab import userdata
child_env['HF_TOKEN']=userdata.get('HF_TOKEN') or ''
command=[sys.executable,'-m','experiments.run_paper_v2','--worker',WORKER,'--mode','formal',
         '--drive-root',str(drive_parent),'--runtime-root',str(runtime_root)]
completed=subprocess.run(command,cwd=checkout,env=child_env,check=False)
output_root=drive_root/'baselines/paper-baseline-gaussian-shading-v2'
state_path=output_root/'job_state.json'
if completed.returncode != 0:
    state=json.loads(state_path.read_text()) if state_path.exists() else {}
    raise RuntimeError(f"worker exited {completed.returncode}: {state.get('error',state.get('status'))}")
final_path=output_root/'method_final.json'
if final_path.exists():
    public=json.loads(final_path.read_text())
    print({'status':public['status'],'result_package_produced':public['result_package_produced'],'output':str(output_root)})
else:
    state=json.loads(state_path.read_text()) if state_path.exists() else {}
    print({'status':state.get('status','WAITING'),'result_package_produced':False,'output':str(output_root)})
